## How to reproduce

3D LYNX run and downstream analysis on the multi-section liver dataset: cell-type
annotation is transferred from `section_08`, LYNX is run jointly across the registered sections
with the 3D prior over the DESI stack, and results are cached under `results/liver_3d/`.
The run reads the data release under `data/LYNX_liver_dataset/NIH_F5_3D/`, whose tables carry
the transferred cell types (see [data/README.md](../../../data/README.md)).

In [ ]:
# --- LYNX reproducibility: repo root + central paths + LYNX import ------------
import os as _os, sys as _sys
try:
    _d = _os.path.dirname(_os.path.abspath(__file__))
except NameError:  # interactive / jupytext: __file__ undefined
    _d = _os.getcwd()
while _d != _os.path.dirname(_d) and not _os.path.exists(_os.path.join(_d, "paths.py")):
    _d = _os.path.dirname(_d)
if _d not in _sys.path:
    _sys.path.insert(0, _d)
from paths import DATA_DIR, RESULTS_DIR, FIGURES_DIR, FINAL_FIGURES_DIR
import lynx  # noqa: F401 - registers models/ & util/ on sys.path
# -----------------------------------------------------------------------------

In [ ]:
import os
import gc
import sys

import numpy as np
import scanpy as sc
import pandas as pd
import anndata as ad
import squidpy as sq
import spatialdata as sd
import spatialdata_io as sdio

import pyro
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib import rcParams

sns.set_context('paper')
rcParams.update({'font.family': 'Arial'})
rcParams.update({'font.size': 12})
rcParams.update({'figure.dpi': 180})
rcParams.update({'savefig.dpi': 300})
import IO, plot, utils, test_assoc, trajectory
import vgae, configs, dataset

from importlib import reload
%matplotlib inline
%load_ext autoreload
%autoreload 2

#### Cell-type annotation

- Source annotation from `section_08`
- Transfer learning w/ `scanpy`

In [ ]:
annot_df = pd.read_csv(DATA_DIR + '/EDITS_liver_55588__subclustered_cell_annotations.csv')
barcodes = set(annot_df['cell_id'].values)
annot_df.set_index('cell_id', inplace=True)
annot_df.head()

In [ ]:
DATA_PATH = DATA_DIR + '/xenium_3d/'
SAMPLE_ID = 'NIH_F5'
SECTIONS = sorted([
   path for path in os.listdir(os.path.join(DATA_PATH, SAMPLE_ID))
   if os.path.isdir(os.path.join(DATA_PATH, SAMPLE_ID, path))
])


In [ ]:
ref_section = 'section_08'
ref_sdata = sd.read_zarr(
    os.path.join(DATA_PATH, SAMPLE_ID, ref_section, 'sdata2.zarr')
)
ref_adata = ref_sdata['table'].copy()

sc.pp.normalize_total(ref_adata, target_sum=1e4)
sc.pp.log1p(ref_adata)
sc.pp.pca(ref_adata)
sc.pp.neighbors(ref_adata)
sc.tl.umap(ref_adata)


In [ ]:
for target_section in SECTIONS[:-1]:
    if os.path.exists(os.path.join(DATA_PATH, SAMPLE_ID, target_section, 'sdata1.zarr')):
        target_sdata = sd.read_zarr(
            os.path.join(DATA_PATH, SAMPLE_ID, target_section, 'sdata1.zarr')
        )
    else:
        target_sdata = sd.read_zarr(
            os.path.join(DATA_PATH, SAMPLE_ID, target_section, 'sdata.zarr')
        )
    target_adata = target_sdata['table'].copy()

    target_adata.obs.set_index('cell_id', inplace=True)
    sc.pp.normalize_total(target_adata, target_sum=1e4)
    sc.pp.log1p(target_adata)
    sc.pp.pca(target_adata)
    sc.pp.neighbors(target_adata)
    sc.tl.umap(target_adata)

    # Celltype transfer
    sc.tl.ingest(target_adata, ref_adata, obs='cell_type')
    sc.pl.umap(ref_adata, color='cell_type', title=ref_section)
    sc.pl.umap(target_adata, color='cell_type', title=target_section)

    # Save updated target sdata
    target_sdata['table'].obs.set_index('cell_id', inplace=True)
    target_adata_tmp = target_sdata['table'][target_adata.obs_names].copy()
    target_adata_tmp.obs['cell_type'] = target_adata.obs['cell_type'].values
    target_adata_tmp.obs['cell_id'] = target_adata.obs_names
    target_sdata['table'] = target_adata_tmp.copy()

    target_sdata.write(os.path.join(DATA_PATH, SAMPLE_ID, target_section, 'sdata2.zarr'), overwrite=True)



#### Run LYNX

In [ ]:
# Hyperparameters
version = 'revised'  # 'original': the published model, w/o the 3D prior
n_subgraphs = 16
n_hidden = 32
n_latent = 6
n_epochs = 500
radius = 50
lr = 1e-2
patience = 20

# Data release: registered sections w/ cross-modal maps & cell types
XENIUM_PATH = DATA_DIR + '/LYNX_liver_dataset/NIH_F5_3D/xenium/'
DESI_PATH = DATA_DIR + '/LYNX_liver_dataset/NIH_F5_3D/DESI/'
SAMPLE_ID = 'NIH_F5'
SDATA_SUFFIX = 'sdata.zarr'

SECTIONS = [
   'section_01',
   'section_02',
   'section_04',
   'section_05',
   'section_06',
   'section_07',
   'section_08',
]

# Sections are numbered in cutting order, 110 um apart (section_03 is absent)
SPACING = 110.
VOXEL = 40.  # In-plane voxel of the DESI stack: the DESI pixel (um)
SEEDS = [0, 1, 2]

cluster_key = 'cell_type'
outdir = RESULTS_DIR + '/liver_3d/'
if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)


def get_depth(section):
    r"""Depth of a section (um)"""
    return SPACING * (int(section.split('_')[-1]) - 1)


se = lambda values: np.std(values, ddof=1) / np.sqrt(len(values))  # Standard error over sections

Joint run on the registered sections w/ the 3D prior: the DESI sections are stacked on their depth grid,
and the prior of each section is its own slice of a 3D convolution over the stack

In [ ]:
def load_section(section):
    r"""Paired tables of a section (cells & DESI pixels mapped onto each other)"""
    adata_xenium = sd.read_zarr(
        os.path.join(XENIUM_PATH, section, SDATA_SUFFIX)
    )['table']
    adata_desi = sd.read_zarr(
        os.path.join(DESI_PATH, section+'_'+SDATA_SUFFIX)
    )['table']
    cell_types = adata_xenium.obs[cluster_key].cat.categories
    adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

    # The filter drops a section's empty cell types: keep the same cell-type codes in every section
    adata_xenium.obs[cluster_key] = pd.Categorical(adata_xenium.obs[cluster_key].astype(str), categories=cell_types)
    return adata_xenium, adata_desi


def run_lynx(sections, held_out=None, seed=0):
    r"""
    Fit LYNX on the given sections & infer their gradient pseudotime (t); several sections are fitted jointly w/ the 3D prior.
    `held_out`: a section that gives neither DESI nor cells to the fit (w/ the 3D prior it keeps its empty place in the stack)
    """
    pyro.set_rng_seed(seed)
    adatas_xenium, adatas_desi = map(list, zip(*[load_section(section) for section in sections]))

    # Configure dataset and model
    stack = dict(
        depths=[get_depth(section) for section in sections], voxel=VOXEL
    ) if len(sections) > 1 and version == 'revised' else {}
    graph_data = dataset.HeteroDataset(
        adatas_ref=adatas_xenium,
        adatas_query=adatas_desi,
        n_subgraphs=n_subgraphs,
        r=radius, is_weighted=True, alpha=0.5,
        cluster_key=cluster_key,
        **stack
    )
    if held_out is not None:
        i = sections.index(held_out)
        if graph_data.volume is not None:
            graph_data.volume[:, adatas_desi[i].obsm['voxel'][0, 0]] = 0
        graph_data.hetero_batches = graph_data.hetero_batches[:n_subgraphs*i] + graph_data.hetero_batches[n_subgraphs*(i+1):]

    train_configs = configs.set_train_configs(
        n_epochs=n_epochs, lr=lr, patience=patience,
        device=torch.device('cuda')
    )

    model_configs = configs.set_model_configs(
        graph_data=graph_data,
        c_hidden=n_hidden,
        c_latent=n_latent,
        act=nn.SiLU(),
        infer_cell_interaction=True,
        version=version
    )

    model = vgae.HeteroAttnVGAE(model_configs, device=torch.device('cuda'))
    model.fit(graph_data, train_configs)

    # Per-sample inference & gradient pseudotime (t)
    for adata_xenium, adata_desi in zip(adatas_xenium, adatas_desi):
        model.evaluate(
            adata_xenium, adata_desi,
            graph_data=graph_data,
            device=torch.device('cuda')
        )
        curve = trajectory.get_curve(adata_desi, epg_lambda=0.01, trim_radius_ratio=0.25)
        trajectory.compute_pseudotime(adata_desi, curve, root_marker=adata_desi.var_names[0])
        curve = trajectory.get_curve(adata_xenium, epg_lambda=0.01, trim_radius_ratio=0.25)
        trajectory.compute_pseudotime(adata_xenium, curve, root_marker='DPT')

    gc.collect()
    return adatas_xenium, adatas_desi


def save_run(sections, adatas_xenium, adatas_desi, tag=''):
    r"""Save adatas w/ inferred pseudotime"""
    for section, adata_xenium, adata_desi in zip(sections, adatas_xenium, adatas_desi):
        adata_desi.write_h5ad(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_{section}_desi{tag}.h5ad'))
        adata_xenium.write_h5ad(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_{section}_xenium{tag}.h5ad'))

In [ ]:
# Joint training
adatas_xenium, adatas_desi = run_lynx(SECTIONS)

Save predictions:

In [ ]:
# Inferred gradient pseudotime (t) per section
for i, section in enumerate(SECTIONS):
    sq.pl.spatial_scatter(adatas_desi[i], color='t', cmap='RdBu_r', size=1, img=False)
    plot.disp_trajectory(adatas_desi[i], cmap='RdBu_r')

    sq.pl.spatial_scatter(adatas_xenium[i], color='t', cmap='RdBu_r', size=20, img=False)
    plot.disp_trajectory(adatas_xenium[i], cmap='RdBu_r')

In [ ]:
# Save adatas w/ inferred pseudotime
save_run(SECTIONS, adatas_xenium, adatas_desi)

#### Joint vs. section-wise analysis

- The graphs stay within a section: the adjacent section is beyond the neighbourhood radius
- DESI used jointly (3D prior over the stack) vs. section by section, three runs each
- A held-out depth: the gradient of a left-out section interpolated from its nearest sections

In [ ]:
# Graph neighbours: other-type cells within the graph radius, within the distance to the adjacent section,
# and DESI pixels of the pixel's own section within that distance (the DESI graph links the 8 nearest)
from scipy.spatial import cKDTree

count_neighbors = lambda coords, r: cKDTree(coords).query_ball_point(coords, r, return_length=True) - 1

records = []
for section in SECTIONS:
    adata_xenium, adata_desi = load_section(section)
    coords = np.asarray(adata_xenium.obsm['spatial'], dtype=float)
    codes = adata_xenium.obs[cluster_key].cat.codes.to_numpy()
    count_others = lambda r: count_neighbors(coords, r).mean() - np.concatenate([
        count_neighbors(coords[codes == c], r) for c in np.unique(codes)
    ]).mean()
    records.append({
        'section': section,
        'cells_50um': count_others(radius),
        'cells_110um': count_others(SPACING),
        'pixels_110um': count_neighbors(np.asarray(adata_desi.obsm['xenium_map'], dtype=float), SPACING).mean()
    })

neighbor_df = pd.DataFrame(records)
neighbor_df.to_csv(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_neighbour_counts.csv'), index=False)
display(neighbor_df.round(1))
print({col: f'{neighbor_df[col].mean():.1f} ± {se(neighbor_df[col]):.1f}' for col in neighbor_df.columns[1:]})

In [ ]:
# Replicate runs of the joint analysis
for seed in SEEDS[1:]:
    save_run(SECTIONS, *run_lynx(SECTIONS, seed=seed), tag=f'_seed{seed}')

# Section-wise analysis: each section fitted alone
for section in SECTIONS:
    for seed in SEEDS:
        save_run([section], *run_lynx([section], seed=seed), tag='_sectionwise'+(f'_seed{seed}' if seed else ''))

In [ ]:
import warnings
from scipy.spatial import cKDTree

def interpolate_gradient(section, coords, neighbors, r=VOXEL):
    r"""
    Gradient of a section interpolated from its neighbors {section: (coords, t)}: at each position, the mean gradient
    of the nearest section on each side (its cells within one DESI pixel); NaN where no cell is that close
    """
    below = [s for s in neighbors if get_depth(s) < get_depth(section)]
    above = [s for s in neighbors if get_depth(s) > get_depth(section)]
    nearest = ([max(below, key=get_depth)] if below else []) + ([min(above, key=get_depth)] if above else [])

    values = []
    for s in nearest:
        neighbor_coords, t = neighbors[s][0], np.asarray(neighbors[s][1], dtype=float)
        values.append(np.array([
            t[indices].mean() if len(indices) else np.nan
            for indices in cKDTree(neighbor_coords).query_ball_point(coords, r=r)
        ]))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return np.nanmean(np.vstack(values), axis=0)

In [ ]:
# Held-out depth: joint analysis w/o one section at a time
for i, section in enumerate(SECTIONS):
    heldout_xenium, heldout_desi = run_lynx(SECTIONS, held_out=section)
    heldout_xenium[i].obs['t_interp'] = interpolate_gradient(
        section, heldout_xenium[i].obsm['spatial'],
        {s: (adata.obsm['spatial'], adata.obs['t']) for s, adata in zip(SECTIONS, heldout_xenium) if s != section}
    )
    save_run([section], [heldout_xenium[i]], [heldout_desi[i]], tag='_heldout')

In [ ]:
# Per section, Spearman over its cells: joint vs. section-wise gradients (every pair of runs), two runs of the same analysis,
# and the section's gradient vs. the one interpolated from its nearest sections (first run of each)
from itertools import combinations
from scipy.stats import spearmanr

rho = lambda u, v: float(spearmanr(u, v, nan_policy='omit')[0])
load_obs = lambda section, tag='': sc.read_h5ad(
    os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_{section}_xenium{tag}.h5ad'), backed='r'
)
t = {
    (run, seed, section): load_obs(section, run+(f'_seed{seed}' if seed else '')).obs['t'].to_numpy(float)
    for run in ['', '_sectionwise'] for seed in SEEDS for section in SECTIONS
}
coords = {section: np.asarray(load_obs(section, '_sectionwise').obsm['spatial']) for section in SECTIONS}

records = []
for section in SECTIONS:
    rho_runs = lambda run: np.mean([rho(t[run, i, section], t[run, j, section]) for i, j in combinations(SEEDS, 2)])
    heldout_obs = load_obs(section, '_heldout').obs
    records.append({
        'section': section,
        'depth_um': get_depth(section),
        'n_cells': len(coords[section]),
        'rho_joint_sectionwise': np.mean([
            rho(t['', i, section], t['_sectionwise', j, section]) for i in SEEDS for j in SEEDS
        ]),
        'rho_runs_joint': rho_runs(''),
        'rho_runs_sectionwise': rho_runs('_sectionwise'),
        'rho_interpolated_sectionwise': rho(
            t['_sectionwise', 0, section],
            interpolate_gradient(section, coords[section], {
                s: (coords[s], t['_sectionwise', 0, s]) for s in SECTIONS if s != section
            })
        ),
        'rho_interpolated_joint': rho(heldout_obs['t'], heldout_obs['t_interp'])
    })

rho_df = pd.DataFrame(records)
rho_df.to_csv(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_joint_vs_sectionwise.csv'), index=False)
display(rho_df.round(3))
print({col: f'{rho_df[col].mean():.3f} ± {se(rho_df[col]):.3f}' for col in rho_df.columns[3:]})

#### Downstream analysis

Load pretrained model results:

In [ ]:
# Load adata w/ inferred spatial gradients
outdir = RESULTS_DIR + '/liver_3d/'
XENIUM_PATH = DATA_DIR + '/LYNX_liver_dataset/NIH_F5_3D/xenium/'
DESI_PATH = DATA_DIR + '/LYNX_liver_dataset/NIH_F5_3D/DESI/'
SAMPLE_ID = 'NIH_F5'
SDATA_SUFFIX = 'sdata.zarr'
SECTIONS = [
   'section_01',
   'section_02',
   'section_04',
   'section_05',
   'section_06',
   'section_07',
   'section_08',
]
cluster_key = 'cell_type'

adatas_desi = []
adatas_xenium = []
for i, section in enumerate(SECTIONS):
    adatas_desi.append(sc.read_h5ad(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_{section}_desi.h5ad')))
    adata_xenium = sc.read_h5ad(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_{section}_xenium.h5ad'))
    sc.pp.normalize_total(adata_xenium, target_sum=1e4)
    sc.pp.log1p(adata_xenium)
    adatas_xenium.append(adata_xenium)

del i, section, adata_xenium
gc.collect()
    

3D visualization:

In [ ]:
import napari

In [ ]:
# z_depths = np.array([
#     int(section.split('_')[-1])
#     for section in SECTIONS
# ])
# if z_depths.min() == 1:
#     z_depths -= 1

z_depths = np.arange(len(SECTIONS))

In [ ]:
# Visualize feature / spatial gradients in 3D

# Create 3D point cloud visualization from adatas with anisotropy

all_coords_3d = []
all_pseudotime = []

# feature = 'CYP3A4'
all_feature = []

anisotropy = 1  # Total layers per section (1 real +  anisotropy-1 fake)

for i, adata in enumerate(adatas_xenium):
    coords_2d = adata.obsm['spatial']
    # Scale z coordinate by anisotropy factor
    z_coord = z_depths[i] * anisotropy
    
    # Create 3D coordinates
    coords_3d = np.column_stack([
        np.full(len(coords_2d), z_coord),  # Z (depth)
        coords_2d[:, 1],  # Y
        coords_2d[:, 0]   # X
    ])
    
    all_coords_3d.append(coords_3d)
    all_pseudotime.append(adata.obs['t'].values)
    # all_feature.append(adata[:, feature].X.toarray().flatten())

# Stack all coordinates and pseudotime values
points = np.vstack(all_coords_3d)
pseudotimes = {'pseudotime': np.hstack(all_pseudotime)}
# features = {'pseudotime': np.hstack(all_feature)}


# Visualize in napari
viewer = napari.Viewer()
viewer.add_points(
    points,
    features=pseudotimes,
    face_color='pseudotime',
    face_colormap='RdBu_r',
    size=25,
    name='Xenium cells 3D'
)

In [ ]:
del viewer
gc.collect()

3D visualization of warped image — multimodal_aligned across all depths

In [ ]:
import napari
from spatialdata.transformations import get_transformation

z_depths = np.array([
    int(section.split('_')[-1])
    for section in SECTIONS
])
if z_depths.min() == 1:
    z_depths -= 1

In [ ]:
# 3D visualization of warped image — multimodal_aligned across all depths
image_key = 'morphology_focus'
anisotropy = 10  # match point cloud spacing

viewer = napari.Viewer()

for i, section in enumerate(SECTIONS):
    sdata_i = sd.read_zarr(
        os.path.join(XENIUM_PATH, section, SDATA_SUFFIX)
    )
    coord_system = 'aligned' if 'aligned' in sdata_i.coordinate_systems else 'global'
    img_aligned = sd.transform(sdata_i[image_key], to_coordinate_system=coord_system)
    img_da = img_aligned['scale0']['image']

    # sd.transform stores the translation as xarray coords (not in pixels);
    # extract Y/X offsets before .to_numpy() strips them
    # y_off = float(img_da.coords['y'].values[0]) if 'y' in img_da.coords else 0.0
    # x_off = float(img_da.coords['x'].values[0]) if 'x' in img_da.coords else 0.0
    arr = img_da.to_numpy()[3]

    z_coord = z_depths[i] * anisotropy
    viewer.add_image(
        arr[np.newaxis],
        name=section,
        colormap='gray',
        opacity=0.75,
        translate=[z_coord, 0, 0],  # full 3D placement: depth + XY alignment
        blending='additive',
    )

    del sdata_i, img_aligned
    gc.collect()

viewer.dims.ndisplay = 3

In [ ]:
del viewer
gc.collect()

phenotype dynamics:

In [ ]:
# TODO: finish if using 3D gradients
sq.pl.spatial_scatter(
    adatas_xenium[-1], 
    color=cluster_key,
    groups=[
        'Pericentral hepatocyte',
        'Periportal hepatocyte',
        # 'Smooth muscle cell',
        # 'Vascular endothelial cell',
        # 'Liver sinusoidal endothelial cell'
    ],
    size=20,
    img=False,
)

gene & metabolite dynamics

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import anndata as ad
from typing import List
import matplotlib as mpl
from scipy.stats import gaussian_kde

def plot_3d_expression_pseudotime(
    adatas: List[ad.AnnData],
    gene: str,
    obs_key: str = 't',
    ratio: float = 1.0,
    cmap: str = 'pink',
    figsize: tuple = (12, 10),
    ax: plt.Axes = None,
    label_fontsize: int = 5,
    tick_fontsize: int = 4,
    zlim: tuple = None,
    elev: float = 20,
    azim: float = 45,
    alpha: float = 0.6,
    s: float = 1,
    plot_kde: bool = True,
    kde_color: str = 'lightsteelblue',
):
    """
    Create a 3D scatter plot without interpolation.
    
    Parameters
    ----------
    adatas : List[ad.AnnData]
        List of AnnData objects, one per depth layer
    gene : str
        Gene name to plot expression for (must be in adata.var_names)
    obs_key : str, default='t'
        Key in adata.obs containing pseudotime values
    ratio : float, default=1.0
        Fraction of cells to randomly sample from each adata (0 < ratio <= 1.0)
    cmap : str, default='viridis'
        Colormap for the scatter plot
    figsize : tuple, default=(12, 10)
        Figure size
    ax : plt.Axes, default=None
        Placeholder for existing subplot ax
    zlim : tuple, optional
        Z-axis limits (depth range)
    elev : float, default=20
        Elevation angle for 3D view
    azim : float, default=45
        Azimuth angle for 3D view
    alpha : float, default=0.6
        Transparency of points
    s : float, default=1
        Size of scatter points
    plot_kde : bool, default=False
        If True, overlay KDE curves for each depth showing expression vs pseudotime
    kde_color : str, default='lightsteelblue'
        Color for KDE shaded area
        
    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object
    ax : matplotlib.axes._subplots.Axes3DSubplot
        The 3D axes object
    """
    
    if not 0 < ratio <= 1.0:
        raise ValueError("ratio must be between 0 and 1")
    
    # Collect data from all depths
    all_expression = []
    all_pseudotime = []
    all_depth = []
    depth_data = {}  # Store raw data for each depth (for KDE)
    
    for depth_idx, adata in enumerate(adatas):
        if gene not in adata.var_names:
            print(f"Warning: {gene} not found in adata at depth {depth_idx}, skipping...")
            continue
        
        if obs_key not in adata.obs.columns:
            print(f"Warning: {obs_key} not found in adata.obs at depth {depth_idx}, skipping...")
            continue
        
        # Get expression values (assuming raw or normalized counts)
        if hasattr(adata, 'raw') and adata.raw is not None:
            expr = adata.raw[adata.obs_names, gene].X
        else:
            expr = adata[:, gene].X
        
        # Handle sparse matrices
        if hasattr(expr, 'toarray'):
            expr = expr.toarray()
        else:
            expr = np.array(expr)
        if expr.ndim > 1:
            expr = expr[:, 0]  # guard against duplicate gene names in var_names
        expr = expr.flatten()
        
        # Get pseudotime values
        pseudotime = adata.obs[obs_key].values
        
        # Store raw data for KDE
        depth_data[depth_idx] = {'expr': expr, 'pseudotime': pseudotime}
        
        # Random sampling
        n_cells = len(adata)
        n_sample = max(1, int(n_cells * ratio))
        sample_idx = np.random.choice(n_cells, size=n_sample, replace=False)
        
        # Store sampled data
        all_expression.extend(expr[sample_idx])
        all_pseudotime.extend(pseudotime[sample_idx])
        all_depth.extend([depth_idx] * n_sample)
    
    # Convert to numpy arrays
    all_expression = np.array(all_expression)
    all_pseudotime = np.array(all_pseudotime)
    all_depth = np.array(all_depth)
    depth_scale = 1.0  # stretch depth axis for visual separation
    all_depth_plot = all_depth * depth_scale
    
    
    # print(f"Total sampled cells: {len(all_expression)}")
    
    # Create 3D plot
    if ax is None:
        fig = plt.figure(figsize=figsize, dpi=300)
        ax = fig.add_subplot(111, projection='3d')
    else:
        fig = ax.get_figure()
    
    # Create scatter plot
    scatter = ax.scatter(
        all_expression, 
        all_pseudotime, 
        all_depth_plot,
        c=all_expression,
        cmap=cmap,
        s=s,
        alpha=alpha,
        edgecolors='none'
    )
        
    # Overlay shaded KDE per depth
    if plot_kde:
        for depth_idx in sorted(depth_data.keys()):
            data = depth_data[depth_idx]
            pseudo = data['pseudotime']
            expr = data['expr']
            pt_min, pt_max = 0, 1
            pt_grid = np.linspace(pt_min, pt_max, 100)
            bw = (pt_max - pt_min) * 0.05  # high bandwidth
            weights = np.exp(-0.5 * ((pseudo[np.newaxis, :] - pt_grid[:, np.newaxis]) / bw) ** 2)
            expr_smooth = (weights * expr[np.newaxis, :]).sum(axis=1) / weights.sum(axis=1)
            z = depth_idx * depth_scale
            polygon = ([(0, pt_grid[0], z)]
                       + [(e, p, z) for e, p in zip(expr_smooth, pt_grid)]
                       + [(0, pt_grid[-1], z)])

            poly = Poly3DCollection(
                [polygon], alpha=0.3,
                facecolor=kde_color,
                edgecolor='none',
            )
            ax.add_collection3d(poly)

            ax.plot(
                expr_smooth, pt_grid, zs=z,
                color='steelblue', linewidth=1.0,
                zorder=z+1
            )
    
    ax.set_xlabel(gene, fontsize=label_fontsize, labelpad=-8)
    ax.set_ylabel(r"Gradient ($t$) ($PV \rightarrow CV$)", fontsize=label_fontsize, labelpad=-8)
    ax.set_zlabel('Depth', fontsize=label_fontsize, labelpad=-8)
    
    # Set axis limits
    if zlim is not None:
        ax.set_zlim(zlim)
    else:
        ax.set_zlim([all_depth_plot.min(), all_depth_plot.max()])
    # Relabel z-ticks to show original depth indices
    n_depths = len(depth_data)
    ax.set_zticks([i * depth_scale for i in range(n_depths)])
    ax.set_zticklabels([str(i) for i in range(n_depths)])
        
    # Remove grid and customize axes to be simpler
    ax.grid(False)
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    
    # Set axis line width
    ax.xaxis.line.set_linewidth(0.5)
    ax.yaxis.line.set_linewidth(0.5)
    ax.zaxis.line.set_linewidth(0.5)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo['tick']['linewidth'] = {True: 0.5, False: 0.5}
    ax.tick_params(axis='both', which='major', labelsize=tick_fontsize, pad=-3)

    # Set viewing angle
    ax.view_init(elev=elev, azim=azim)
    
    # Add smaller colorbar
    cbar = fig.colorbar(scatter, ax=ax, shrink=0.5, aspect=20, pad=0.05)
    cbar.set_label(r'Expression', size=label_fontsize)
    cbar.ax.tick_params(labelsize=tick_fontsize)

    return fig, ax

In [ ]:
# [DEBUG]: temporary DESI annotation map by assigning closest m/z
# Load desi annotation
def is_float(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

# Cleanup annotation map
mz_annot_df = pd.read_csv(DATA_DIR + '/DESI_annotation_3D.csv', usecols=[0,1])
mz_annot_dict = dict(
    (k, v) for k, v in zip(
        mz_annot_df['mz'].apply(lambda x: x.split('m/z')[0].strip()).values, 
        mz_annot_df['Annotation']
    ) if is_float(k) and str(v) != 'nan'
)

# Map from anndata m/z values with closest assignment
epsilon = 1e-1
query_mz = pd.Series(adatas_desi[0].var_names).apply(lambda x: x.split('m/z')[0].strip())
query_dict = {mz: v for v, mz in zip(adatas_desi[0].var_names, query_mz)}

mz_assignments = {}
for query in query_mz:
    matches = pd.Series(mz_annot_dict.keys()).apply(lambda x: np.abs(float(x) - float(query)))
    if matches.min() < epsilon:
        # print(key, query)
        idx = matches.argmin()
        key = pd.Series(mz_annot_dict.keys()).iloc[idx]
        mz_assignments[query_dict[query]] = mz_annot_dict[pd.Series(mz_annot_dict.keys()).iloc[idx]]

# In-place assignmennt to adatas (NOTE: don't save before validations)
for i in range(len(adatas_desi)):
    adatas_desi[i].var_names = adatas_desi[i].var_names.map(lambda x: mz_assignments.get(x, x)) 

del matches, idx, key, query, epsilon

In [ ]:
features = [
    'PC 32:0',
    'CAR 18:2',
    'PE 40:4',
    'PC 36:3'
]

fig, axes = plt.subplots(
    1, 4, figsize=(24, 7), dpi=300, 
    subplot_kw={'projection': '3d'},
    layout="constrained"
)
for i, feature in enumerate(features):
    _ = plot_3d_expression_pseudotime(
        adatas=adatas_desi,
        gene=feature,
        obs_key='t',
        cmap='magma',
        ratio=0.25,
        figsize=(6, 7),
        ax=axes[i],
        elev=40,
        azim=300,
        alpha=0.75,
        label_fontsize=10,
        tick_fontsize=6,
        s=1,
        plot_kde=True,
        kde_color='#D5E8D4',
    )
plt.show()
fig.savefig(os.path.join(FIGURES_DIR + '/', f'LYNX_Fig5_metabolite_3d.svg'), bbox_inches='tight', dpi=300)
gc.collect()

In [ ]:
features = [
    'MYH11',
    'HAMP',
    'LYVE1',
    'CYP3A4'
]

fig, axes = plt.subplots(
    1, 4, figsize=(24, 7), dpi=300, 
    subplot_kw={'projection': '3d'},
    layout="constrained"
)
for i, feature in enumerate(features):
    _ = plot_3d_expression_pseudotime(
        adatas=adatas_xenium,
        gene=feature,
        obs_key='t',
        cmap='magma',
        ratio=0.1,
        figsize=(6, 7),
        ax=axes[i],
        elev=40,
        azim=300,
        alpha=0.75,
        label_fontsize=10,
        tick_fontsize=6,
        s=1,
        plot_kde=True
    )
plt.show()
fig.savefig(os.path.join(FIGURES_DIR + '/', f'LYNX_Fig5_gene_3d.svg'), bbox_inches='tight', dpi=300)
gc.collect()

Summary statistics:

In [ ]:
obs_key = 't'
records = []
for depth_idx, (ad_d, ad_x) in enumerate(zip(adatas_desi, adatas_xenium)):
    for t in ad_d.obs[obs_key].dropna():
        records.append({'depth': depth_idx, 'modality': 'DESI', obs_key: t})
    for t in ad_x.obs[obs_key].dropna():
        records.append({'depth': depth_idx, 'modality': 'Xenium', obs_key: t})

df = pd.DataFrame(records)

# Violin plot
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
sns.violinplot(
    data=df[::-1], x=obs_key, y='depth', hue='modality', split=True, inner=None,
    palette={'DESI': '#4C72B0', 'Xenium': '#DD8452'},
    linewidth=0.6, 
    orient='h',
    ax=ax
)
ax.set_ylabel('Depth (section index)', fontsize=8)
ax.set_yticklabels([8,7,6,5,4,2,1], fontsize=8)
ax.set_xlabel(r'Gradient coordinate ($t$)'+'\n'+r'$(PV \rightarrow CV$)', fontsize=8)
ax.set_title('Gradient coordinate distribution across depths', fontsize=10)
ax.legend(title='Modality', fontsize=6, title_fontsize=8, loc='upper right', bbox_to_anchor=(1.2, 1.0))
ax.tick_params(labelsize=8)
sns.despine()
plt.tight_layout()

fig.savefig(os.path.join(FIGURES_DIR + '/', f'LYNX_Fig5_pseudotime_violin.svg'), bbox_inches='tight')


# Kruskal-Wallis test (depth invariance within each modality)
# for modality in ['DESI', 'Xenium']:
#     groups = [
#         df.loc[(df['modality'] == modality) & (df['depth'] == d), obs_key].values
#         for d in sorted(df['depth'].unique())
#     ]
#     H, p = kruskal(*groups)
#     print(f'{modality:8s}  Kruskal-Wallis H = {H:.3f},  p = {p:.3e}  '
#           f'({"NOT " if p > 0.05 else ""}significant at α=0.05)')
